The ``aaanalysis.pipe`` (``aap``) module provides high-level **golden pipelines** — stateless, one-call wrappers over the AAanalysis primitives. ``aap.predict`` takes a feature set (``df_feat``) and the sequence parts (``df_parts``), rebuilds the feature matrix, fits a ``TreeModel``, and returns the fitted model together with a cross-validated evaluation. Its defaults are byte-identical to writing the explicit ``feature_matrix`` → ``TreeModel.fit`` → ``TreeModel.eval`` chain by hand.

We first load the ``DOM_GSEC`` example dataset, a small feature set, and the sequence parts (see [Breimann25]_):

In [1]:
import aaanalysis as aa
import aaanalysis.pipe as aap
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=20)
labels = df_seq["label"].to_list()
df_feat = aa.load_features().head(8)
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)

aa.display_df(df_feat, n_rows=10, show_shape=True)

DataFrame shape: (8, 15)


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions,feat_importance,feat_importance_std
1,"TMD_C_JMD_C-Seg...3,4)-KLEP840101",Energy,Charge,Charge,"Net charge (Kle...n et al., 1984)",0.244000,0.103666,0.103666,0.106692,0.110506,0.000000,0.000000,"31,32,33,34,35",0.970400,1.438918
2,"TMD_C_JMD_C-Seg...3,4)-FINA910104",Conformation,α-helix (C-cap),α-helix termination,"Helix terminati...n et al., 1991)",0.243000,0.085064,0.085064,0.098774,0.096946,0.000000,0.000000,"31,32,33,34,35",0.000000,0.000000
3,"TMD_C_JMD_C-Seg...6,9)-LEVM760105",Shape,Side chain length,Side chain length,"Radius of gyrat... (Levitt, 1976)",0.233000,0.137044,0.137044,0.161683,0.176964,0.000000,0.000001,"32,33",1.554800,2.109848
4,"TMD_C_JMD_C-Seg...3,4)-HUTJ700102",Energy,Entropy,Entropy,"Absolute entrop...Hutchens, 1970)",0.229000,0.098224,0.098224,0.106865,0.124608,0.000000,0.000001,"31,32,33,34,35",3.111200,3.109955
5,"TMD_C_JMD_C-Seg...6,9)-RADA880106",ASA/Volume,Volume,Accessible surface area (ASA),"Accessible surf...olfenden, 1988)",0.223000,0.095071,0.095071,0.114758,0.132829,0.000000,0.000002,"32,33",0.000000,0.000000
6,"TMD_C_JMD_C-Seg...2,3)-KLEP840101",Energy,Charge,Charge,"Net charge (Kle...n et al., 1984)",0.222000,0.058671,0.058671,0.064895,0.069547,0.000000,0.000001,"27,28,29,30,31,32,33",0.000000,0.000000
7,"TMD_C_JMD_C-Seg...4,5)-FAUJ880109",Energy,Isoelectric point,Number hydrogen bond donors,"Number of hydro...e et al., 1988)",0.215000,0.146661,0.146661,0.174609,0.188034,0.000000,0.000004,"33,34,35,36",1.032400,1.510722
8,"TMD_C_JMD_C-Seg...3,4)-JANJ780101",ASA/Volume,Accessible surface area (ASA),ASA (folded protein),"Average accessi...n et al., 1978)",0.215000,0.124317,0.124317,0.166309,0.153364,0.000000,0.000004,"31,32,33,34,35",1.080400,1.296094


A single call returns the fitted ``model`` and the evaluation ``df_eval``. Use ``random_state`` for reproducibility:

In [2]:
model, df_eval = aap.predict(df_feat, df_parts, labels=labels, random_state=42)

aa.display_df(df_eval, n_rows=10, show_shape=True)

/Users/stephanbreimann/Programming/1Packages/aa-wt-pipe/aaanalysis/feature_engineering/_backend/cpp_run.py:163: UserWarning: CPP is using the Python kernel fallback — the compiled Cython extension is not available in this install. Output is bit-exact with the Cython path but ~2x slower. Reinstall via `pip install --force-reinstall aaanalysis` to fetch a prebuilt wheel.
  warnings.warn(


DataFrame shape: (1, 5)


,name,accuracy,precision,recall,f1
1,Set 1,0.737500,0.815000,0.650000,0.699600


The tree-based models (``list_model_classes``), the number of cross-validation folds (``n_cv``), the CPU cores for building the feature matrix (``n_jobs``), and verbosity (``verbose``) can all be set explicitly:

In [3]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

model, df_eval = aap.predict(df_feat, df_parts, labels=labels,
                             list_model_classes=[RandomForestClassifier, ExtraTreesClassifier],
                             n_cv=4, random_state=42, n_jobs=1, verbose=False)

aa.display_df(df_eval, n_rows=10, show_shape=True)

DataFrame shape: (1, 5)


,name,accuracy,precision,recall,f1
1,Set 1,0.762500,0.878600,0.675000,0.716400


The returned ``model`` is a fitted ``TreeModel``, so its feature importances (and any other ``TreeModel`` method) are available for downstream use:

In [4]:
print(f"{type(model).__name__} fitted with {len(model.feat_importance)} feature importances")

TreeModel fitted with 8 feature importances
